In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import joblib
import pandas as pd


In [ ]:
df_combined = pd.read_excel("bee_data_with_features.xlsx")


In [ ]:
# Select only MFCC features
all_features = df_combined[
    ["hive temp", "hive humidity", "weather temp", "weather humidity"] +
    [f"mfcc_{i}" for i in range(1, 14)]
]
# Remove any missing values
all_features = all_features.dropna()

# Fit Isolation Forest
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
df_combined['anomaly'] = iso_forest.fit_predict(all_features)

# Anomalies are marked as -1, normal as 1
df_combined['anomaly_label'] = df_combined['anomaly'].map({1: "Normal", -1: "Anomaly"})

# Count anomalies
print(df_combined['anomaly_label'].value_counts())

In [ ]:
# Reduce to 2D for visualization
pca = PCA(n_components=2)
components = pca.fit_transform(all_features)

plt.figure(figsize=(10, 6))
plt.scatter(components[:, 0], components[:, 1], c=df_combined['anomaly'], cmap='coolwarm', edgecolor='k')
plt.title('Anomaly Detection in Bee Buzzing Sounds (MFCCs)')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.show()


In [ ]:
anomalous_files = df_combined[df_combined['anomaly_label'] == "Anomaly"]['file name'].tolist()
print("Anomalous Audio Files:", anomalous_files)


In [ ]:
joblib.dump(iso_forest, 'bee_sound_anomaly_model.pkl')